# Predicting the Success of a Kickstarter Project

## Import libraries

In [ ]:
import os, json
import pandas as pd
import numpy as np
import warnings
import datetime
import pickle

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.calibration import cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import fbeta_score, make_scorer
from sklearn.metrics import roc_auc_score, roc_curve, RocCurveDisplay
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import EDA_plots as EDA
import evaluate_models as em
import pipeline as pipe

warnings.filterwarnings("ignore")

RSEED = 1312

---

## Prepare data

### Import dataframe

Drop coluns which are "outcome" features like _number of backers_ or _USD pledged_, because these reflect the success of a project and can not be used to try to predict that outcome.

In [ ]:
# --- import dataframe ---
df = pd.read_csv(os.path.join("data", "df_preprocessed.csv"))

In [ ]:
# get rid of "outcome" features & create new df
df = df.drop(
    columns=[
        "backers_count",
        "created_at",
        "deadline_at",
        "launched_at",
        "staff_pick",
        "usd_pledged",
        "pledge_per_backer",
    ],
    axis=1,
)

# target variable
df = df.replace({"state": {"failed": 0, "successful": 1}})
df = df.rename(columns={"state": "outcome"})

# prepare some variables
target = "outcome"
target_labels = ["failed", "successful"]
cat_feat = df.select_dtypes(exclude=["number"]).columns.to_list()
num_feat = df.select_dtypes(include=["number"]).columns.to_list()
num_feat.remove(target)

# print infos
print("Numerical features are: ", num_feat)
print("Categorical features are: ", cat_feat)
df.info()

### Log transformation

Some of the numerical features are highly skewed and will be log transformed.
Features to be log-transformed are `usd_goal`, `name_length`, and `preparation_days`.

In [ ]:
# --- show distributions ---
EDA.histplots(df[num_feat], bins="doane", cols=3)
df.describe().round(0).T

In [ ]:
# --- apply log transformation ---
for f in ["usd_goal", "name_length", "preparation_days"]:
    df[f] = df[f].apply(lambda x: np.log(x + 1))

EDA.histplots(df[num_feat], bins="doane", cols=3)

### Train-Test-Split

In [ ]:
# define predictors and target variable
X = df
y = X.pop(target)

print(f"We have {X.shape[0]} observations in our dataset and {X.shape[1]} features")
print(f"Our target vector has also {y.shape[0]} values")

# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RSEED
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

---

## Baseline model

Looking at some figures created during the EDA, it becomes obvious that some features differ between successful and failed projects. Based on my observations, I'll predict a project as successful, if the follwoing criteria are met:

- category does not belong to `Crafts`, `Photography`, `Technology`, `Journalism` or `Food`, since they have a lower as 50% successrate
- AND 
  - `prepartion time` longer as `median(preparation time)`
    - _OR_
  - `usd goal` lower as `median(usd goal)`
    - _because_ either they prepare very well OR the target is low enough to be achieved easily

In [ ]:
# find rows with mathing features
baseline_idx = X_test[
    (
        ~X_test["category"].isin(
            ["Crafts", "Photography", "Technology", "Journalism", "Food"]
        )
    )
    & (
        (X_test["preparation_days"] > X_train["preparation_days"].median())
        | (X_test["usd_goal"] < X_train["usd_goal"].median())
    )
].index

# create prediction pd.series
baseline_pred = pd.Series(np.zeros((X_test.shape[0], )), index=X_test.index)
baseline_pred[baseline_idx] = 1.0

# report results
em.report(y_test, baseline_pred)

---

## Pipelines

Here, i'll prepare sckit-learn pipelines which will be used in the following classifications.

### Set default parameters

As the metric to be optimized i decided for a f_beta score with beta=0.5, which puts more weight on precision. While avoiding false negatives (missing chances of successful projects) is also a desirable outcome, it is more important to achieve a high precision and avoif false positives (loosing time/money in unsuccessful projects).

In [ ]:
# defaults
cv = 5
n_jobs = -1
verbose = 0

# metric
fhalf_scorer = make_scorer(fbeta_score, beta=0.5)

# overwrite models
overwrite = False  # choose to overwrite files
add_suffix = False  # OR add suffix, if filename already exists

### pipe: preproccessor

In [ ]:
# preprocessor for numerical & categorical features
num_pipeline = Pipeline([("std_scaler", StandardScaler())])
cat_pipeline = Pipeline([("1hot", OneHotEncoder(handle_unknown="ignore"))])

# Complete pipeline for numerical and categorical features
# 'ColumnTransformer' applies transformers (num_pipeline/ cat_pipeline)
# to specific columns of an array or DataFrame (num_features/cat_features)
preprocessor = ColumnTransformer(
    [("num", num_pipeline, num_feat), ("cat", cat_pipeline, cat_feat)]
)

### pipe: Logistic Regression

In [ ]:
# logisitic regression pipeline
LR_pipe = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "logreg",
            LogisticRegression(
                max_iter=1000, random_state=RSEED, n_jobs=n_jobs, verbose=verbose
            ),
        ),
    ]
)

---

## Logistic Regression

### cross_validation of train set

Perform a cross-validation on the training set to get a sense of how well logistic regression is able to find patterns wihti the data and predict the outcome.

In [ ]:
# run cross-validation
LR_y_train_pred_cv = cross_val_predict(
    LR_pipe, X_train, y_train, cv=cv, n_jobs=n_jobs, verbose=verbose
)

# report results
em.report(y_train=y_train, y_train_pred=LR_y_train_pred_cv)

#### Optimizing via Grid Search

In [ ]:
# define parameter grid
param_logreg = {
    "logreg__penalty": ("l1", "l2"),
    "logreg__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "logreg__solver": ["liblinear", "newton-cg", "lbfgs", "sag", "saga"],
    "logreg__class_weight": [None, "balanced"],
}

# intantiate GridSearchCV
LR_gs = GridSearchCV(
    LR_pipe,
    param_grid=param_logreg,
    cv=cv,
    scoring=fhalf_scorer,
    verbose=verbose,
    n_jobs=n_jobs,
)

In [ ]:
# fit GridSearchCV
model_name = "LR_gs.pickle"
if (
    not os.path.exists(os.path.join("models", f"{model_name}"))
    or overwrite
    or add_suffix
):
    LR_gs.fit(X_train, y_train)
    em.save_model(LR_gs, model_name)
else:
    LR_gs = pickle.load(open(os.path.join("models", f"{model_name}"), "rb"))

# display pipeline
LR_gs

In [ ]:
# evaluate GridSearch results
y_test_predicted = em.eval_grid_search(LR_gs, X_test)
em.report(y_test=y_test, y_test_pred=y_test_predicted)